# Databricks Lakehouse — Crash Course

> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)

Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.
Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.

## Hogyan futtasd

```bash
# 1. Virtuális környezet (Python 3.10+)
python -m venv .venv

# Windows:
.venv\Scripts\activate

# macOS/Linux:
source .venv/bin/activate

# 2. Telepítsd a függőségeket (a notebook első cellája)

# 3. Indítsd a Jupytert
jupyter lab
# vagy
jupyter notebook
```

Minden cella saját magában értelmezhető. A `# %%` kommentek Jupyterben és VS Code-ban is a cellák határát jelölik.


## Fontos megjegyzés

A Databricks platform natív futtatási környezete a saját workspace notebookja.

Ez a helyi notebook:

- mutatja a **Databricks-specifikus kódmintát**, amit DBR-ben futtatnál;
- **PySpark + Delta Lake** lokális szimulációval megmutatja ugyanazt a gondolkodást;
- Unity Catalogot és Jobs/DLT funkciókat csak fogalmi szinten érint.


In [ ]:
%pip install pyspark==3.5.3 delta-spark==3.2.* --quiet

## 1. Unity Catalog háromszintű névtér

Databricks Unity Catalog: `<catalog>.<schema>.<table>` — pl. `prod.silver.orders`.**Lokálisan** Spark-ban ezt szimulálhatjuk a `catalog/database/table` konvencióval a fájlrendszeren.


In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('Databricks-Local')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .master('local[*]')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('OK:', spark.version)


## 2. Delta Lake — Medallion architektúra


In [ ]:
from pathlib import Path

Path('lake').mkdir(exist_ok=True)

# Bronze — nyers ingest
bronze = spark.createDataFrame(
    [
        (1, 1, '12500', 'paid', '2025-01-20'),
        (2, 1, '8,900', 'paid', '2025-02-15'),  # szándékos: vesszős szám
        (3, 2, '24000', 'PENDING', '2025-03-01'),
        (4, None, '5200', 'paid', '2025-03-15'),  # szándékos: NULL customer
    ],
    ['order_id', 'customer_id', 'amount_raw', 'status_raw', 'order_date_raw'],
)

bronze.write.format('delta').mode('overwrite').save('lake/bronze/orders')
print('Bronze kiírva')


## 3. Silver — tisztítás, DQ


In [ ]:
from pyspark.sql import functions as F

silver = (
    spark.read.format('delta').load('lake/bronze/orders')
    .filter(F.col('customer_id').isNotNull())
    .withColumn('amount', F.regexp_replace('amount_raw', ',', '').cast('decimal(10,2)'))
    .withColumn('status', F.lower('status_raw'))
    .withColumn('order_date', F.to_date('order_date_raw'))
    .select('order_id', 'customer_id', 'amount', 'status', 'order_date')
)

silver.write.format('delta').mode('overwrite').save('lake/silver/orders')
silver.show()


## 4. Gold — aggregátum riport


In [ ]:
gold = (
    spark.read.format('delta').load('lake/silver/orders')
    .filter(F.col('status') == 'paid')
    .groupBy('customer_id')
    .agg(F.count('*').alias('orders'), F.sum('amount').alias('revenue'))
    .orderBy(F.desc('revenue'))
)

gold.write.format('delta').mode('overwrite').save('lake/gold/customer_kpi')
gold.show()


## 5. Time travel


In [ ]:
# Írjuk felül az adatot — új verzió születik.
silver.filter(F.col('amount') > 10000).write.format('delta').mode('overwrite').save('lake/silver/orders')

# Régi verzió olvasása.
v0 = spark.read.format('delta').option('versionAsOf', 0).load('lake/silver/orders')
v1 = spark.read.format('delta').option('versionAsOf', 1).load('lake/silver/orders')

print('v0 sorok:', v0.count())
print('v1 sorok:', v1.count())


## 6. Databricks-specifikus sample (NE futtasd lokálisan!)

```python
# Csak Databricks workspace-ben:
spark.sql('CREATE CATALOG IF NOT EXISTS prod')
spark.sql('CREATE SCHEMA IF NOT EXISTS prod.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS prod.gold')

df.write.mode('overwrite').saveAsTable('prod.silver.orders')
```


In [ ]:
spark.stop()

## Következő lépések

- Térj vissza a [web-alapú kurzushoz](./index.html) a teljes anyagért, diagramokért és kvízekért.
- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.
- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)

---

*Engineering Crash Courses · MIT License · Magyar Data & AI Engineering kurzusok*
